In [0]:
df_silver_cpt = spark.read.table("he_prod_incen_analytics_dbw_01.silver.dim_cpt_codes")

In [0]:
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

# 1. Define window specification ordered by the business key
window_spec = Window.orderBy("cpt_code")

# 2. Generate Surrogate Key: AUTO_INCREMENT
df_gold_cpt = df_silver_cpt.withColumn("cpt_sk", row_number().over(window_spec))

# 3. Select ONLY the columns required in Gold (Column Pruning & Reordering)
df_gold_cpt = df_gold_cpt.select(
    "cpt_sk",
    "cpt_code",            # Business Key
    "cpt_description",     # Attribute
    "cpt_category",        # Hierarchy Attribute
    "base_price",          # Measure/Attribute (DECIMAL)
    "reimbursement_rate",  # Measure/Attribute (DECIMAL)
    "expected_charge"      # Measure/Attribute (DECIMAL)
)

display(df_gold_cpt.limit(5))

In [0]:
gold_table_fqn = "he_prod_incen_analytics_dbw_01.gold.dim_cpt_codes"

df_gold_cpt.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Dim_CPT_Codes to Gold layer.")